# Champions League + MLflow en Databricks
Andy Fuentes · Diederich Solís · Diego Linares · Christian Echeverria

Adaptación del ZIP y del [Taller 3 original](https://github.com/DiegoLinares11/MLFlow). [Repositorio de esta entrega](https://github.com/Andyfer004/Ejercicio5-MLOPS). Las métricas del notebook 03 son históricas del taller. La ejecución real de ambos formatos se documenta al final con evidencia suministrada por el usuario.

## 1. Entorno
Instale las dependencias según el [README](../README.md). En Databricks use el Git folder de **Andyfer004/Ejercicio5-MLOPS** y un compute Python. Si instala o reinicia Python, vuelva a ejecutar la celda de rutas siguiente: añade la raíz y `src/`, sin depender de que sobreviva la instalación editable.

La raíz se busca desde la ubicación conocida del notebook y desde el directorio actual. Si serverless usa un directorio distinto, indique en `RUTA_WORKSPACE` la ruta absoluta real de este repositorio, por ejemplo `/Workspace/Users/fue22944@uvg.edu.gt/Ejercicio5-MLOPS` (confirme la ruta de su Git folder). No es la ruta del experimento.


In [ ]:
from pathlib import Path
import sys

# Ruta explícita opcional para serverless; use la raíz real de este repositorio.
RUTA_WORKSPACE = ""

ubicaciones = []
for nombre in ("__vsc_ipynb_file__", "__file__"):
    if globals().get(nombre):
        ubicaciones.append(Path(globals()[nombre]).expanduser())
# Contexto disponible en un notebook Databricks; no cambia permisos ni ejecuta runs.
if "dbutils" in globals():
    try:
        ruta_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        ubicaciones.append(Path("/Workspace") / ruta_notebook.lstrip("/"))
    except Exception:
        pass  # La ruta explícita y el directorio actual siguen disponibles.
ubicaciones.append(Path.cwd())

if RUTA_WORKSPACE.strip():
    candidatas = [Path(RUTA_WORKSPACE).expanduser().resolve()]
else:
    candidatas = []
    for ubicacion in ubicaciones:
        ubicacion = ubicacion.resolve()
        inicio = ubicacion.parent if ubicacion.suffix == ".ipynb" or ubicacion.is_file() else ubicacion
        candidatas.extend((inicio, *inicio.parents))
RAIZ = next((ruta for ruta in candidatas
             if (ruta / "pipeline_champions_databricks.py").is_file()
             and (ruta / "src" / "taller3_pipeline").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Configure RUTA_WORKSPACE con la raíz real de Ejercicio5-MLOPS.")
for ruta in (RAIZ, RAIZ / "src"):
    valor = str(ruta)
    if valor in sys.path:
        sys.path.remove(valor)
    sys.path.insert(0, valor)
from pipeline_champions_databricks import ejecutar
from taller3_pipeline.modelos import FAMILIAS
print("Familias disponibles:", list(FAMILIAS))


## 2. Comprensión y preparación de datos (CRISP-DM)
Se eliminan filas vacías, score y winner; el pipeline transforma porcentajes, razones y equipos dentro de cada fold. Las estadísticas de juego son posteriores al encuentro: este ejercicio clasifica resultados con estadísticas observadas y no demuestra capacidad de pronóstico previo.

In [ ]:
from taller3_pipeline.data import extraer_datos, filtrar_datos, cargar_datos
crudos = extraer_datos()
limpios = filtrar_datos(crudos)
X_train, X_test, y_train, y_test = cargar_datos()
print({"filas_crudas": len(crudos), "partidos": len(limpios),
       "entrenamiento": len(X_train), "prueba": len(X_test)})
print(y_train.value_counts())

## 3. Configuración
Configuración de las ejecuciones completas suministradas por el usuario: Databricks, SVC, 40 iteraciones y cinco folds. `MODELO_UC = None`: se guardan MLflow Models sin registrar versiones, alias ni endpoints. Para una prueba local futura, cambie explícitamente `URI` y `EXPERIMENTO` con las alternativas comentadas. Esa prueba creará un run local distinto de la evidencia histórica.


In [ ]:
URI = "databricks"
EXPERIMENTO = "/Users/fue22944@uvg.edu.gt/Champions_Resultado_Databricks"
FAMILIA = "svc"
N_ITER = 40
FOLDS = 5
MODELO_UC = None

# Modo local explícito (opcional; no usar para atribuir resultados a Databricks):
# URI = "sqlite:///" + str(RAIZ / "mlflow.db")
# EXPERIMENTO = "Champions_Local_Notebook"


## 4. Modelado, evaluación y MLflow Models
Experiments organiza los runs; Tracking guarda parámetros, métricas, datasets y artefactos. Models empaqueta el pipeline completo con firma y código. Se recarga el modelo y se comprueba su F1 de prueba. La selección de hiperparámetros usa F1 macro de validación cruzada, sin ajustar sobre el conjunto de prueba.

In [ ]:
resultado = ejecutar(EXPERIMENTO, familia=FAMILIA, n_iter=N_ITER, folds=FOLDS,
                     modelo_uc=MODELO_UC, uri=URI, formato="notebook")
assert resultado["recarga_verificada"]
resultado

### Prueba del archivo Python en el mismo entorno
La ejecución documentada probó el punto de entrada del `.py` con `runpy` como `__main__`, dentro del notebook; no fue un Job independiente. Esta celda permite repetir ese procedimiento de forma explícita, pero no se ha ejecutado durante la preparación del cierre. Las salidas originales no están disponibles en un notebook exportado; consulte la evidencia suministrada al final.


In [ ]:
import runpy
import sys

argumentos_previos = sys.argv[:]
try:
    sys.argv = [
        "pipeline_champions_databricks.py",
        "--uri", URI,
        "--experimento", EXPERIMENTO,
        "--familia", FAMILIA,
        "--n-iter", str(N_ITER),
        "--folds", str(FOLDS),
    ]
    runpy.run_path(str(RAIZ / "pipeline_champions_databricks.py"), run_name="__main__")
finally:
    sys.argv = argumentos_previos


## 5. Resultados y conclusiones

Se ejecutó el pipeline de Champions League en **Databricks** en dos formatos: Jupyter Notebook y archivo Python. Ambos utilizaron SVC con `RandomizedSearchCV`, 40 iteraciones y validación cruzada estratificada de cinco particiones, optimizando F1 macro.

| Métrica | Notebook | Archivo Python |
|---|---:|---:|
| F1 macro de validación cruzada | 0.637570 | 0.637570 |
| Desviación estándar del F1 de validación | 0.051826 | 0.051826 |
| Exactitud en prueba | 0.655172 | 0.655172 |
| F1 macro en prueba | 0.601010 | 0.601010 |
| F1 macro de referencia | 0.217054 | 0.217054 |
| Recarga del modelo verificada | Sí | Sí |

El conjunto contiene 144 partidos válidos: 115 para entrenamiento y 29 para prueba. Los hiperparámetros seleccionados fueron `C=0.17535949529764422`, `class_weight="balanced"`, `kernel="linear"` y `gamma=0.04609877941534893`. Con el kernel lineal, `gamma` no interviene en la decisión del clasificador.

**Trazabilidad de las ejecuciones**

- Notebook: `bfe4871fb7324ad8953efb6bb7fa33cb`.
- Modelo del notebook: `models:/m-39d842fe62224cad9b83571b60c45968`.
- Archivo Python: `79e0d6ade66c4631b2cb47482a6b49f3`.
- Modelo del archivo Python: `models:/m-fa7ce8e782ae498fb1a3f6bdecababdb`.

La igualdad de métricas confirma consistencia entre ambos formatos bajo la misma configuración y partición. El F1 macro de prueba supera la referencia de clase mayoritaria. La selección se basa en validación cruzada; los datos de prueba se utilizan para la evaluación final. El conjunto es pequeño y emplea estadísticas del encuentro, por lo que estos resultados no demuestran pronóstico previo al partido.

Se utilizaron **Experiments, Tracking, Dataset Tracking y MLflow Models**. Cada ejecución guardó parámetros, métricas, datasets, artefactos y el pipeline completo; la recarga reprodujo su F1 macro de prueba. `version=null` indica que no se registró una versión en Model Registry: el modelo sí está guardado en MLflow. Esta ejecución no incluye Feature Store, registro en Unity Catalog ni un endpoint de Model Serving.

### Enlaces de la entrega

- [Repositorio de GitHub](https://github.com/Andyfer004/Ejercicio5-MLOPS)
- [Notebook en el workspace de Databricks](https://dbc-5490c6a5-5215.cloud.databricks.com/editor/notebooks/2665096834519332?o=7474655525361249)
- [Run del notebook](https://dbc-5490c6a5-5215.cloud.databricks.com/ml/experiments/2665096834519373/runs/bfe4871fb7324ad8953efb6bb7fa33cb?o=7474655525361249)
- [Run del archivo Python](https://dbc-5490c6a5-5215.cloud.databricks.com/ml/experiments/2665096834519373/runs/79e0d6ade66c4631b2cb47482a6b49f3?o=7474655525361249)

El acceso a los recursos de Databricks depende de los permisos del workspace.

### Procedencia de la evidencia publicada

La ejecución real se documenta con el [JSON suministrado](../reporte/evidencias/resultados_databricks.json) y las [capturas del usuario](../reporte/evidencias/). Este repositorio no recibió el notebook ejecutado exportado; por ello sus celdas de código conservan salidas vacías y no se asignan contadores de ejecución ni logs artificiales. Los resultados anteriores son evidencia suministrada por el usuario y observada por este en Databricks, no una ejecución remota realizada durante este cierre.
